# PS-Q2: Quantum Kernel Classifier & Hardware Transpilation
## Qiskit Fall Fest 2026 – Track 4 (Quantum Machine Learning)

This notebook implements an end-to-end Quantum Support Vector Machine (QSVM) pipeline and hardware-aware transpilation benchmark adhering strictly to the **PS-Q2** specification.

### Pipeline Structure
1. **Section 1: Dataset Generation & Bloch Sphere Scaling**: `make_moons` with stratified 80/20 split and $[-\pi, \pi]$ scaling.
2. **Section 2: Quantum Feature Mapping & Kernel Estimation**: Parameterized `ZZFeatureMap` (linear vs. circular entanglement) with exact statevector fidelity kernel simulation.
3. **Section 3: Kernel Concentration & Variance Analysis**: Off-diagonal kernel statistics, Seaborn heatmap, and similarity distribution histogram.
4. **Section 4: Model Training & Classical Baselines Comparison**: QSVM vs. Classical RBF (GridSearchCV tuned), Polynomial ($d=3$), and Linear SVMs, including side-by-side decision boundary visualization.
5. **Section 5: Hardware-Aware Transpilation & SWAP Routing Overhead**: Transpilation across optimization levels 0–3 on Processor A (5Q linear chain) and Processor B (9Q branched heavy-hex), CX and SWAP overhead analysis.
6. **Section 6: Graph Topology Analysis & Benchmark Metrics**: NetworkX graph invariant calculation (diameter, avg path length, avg degree) and unified benchmark export.


In [1]:
# ============================================================
# Environment Setup & Core Imports
# ============================================================
import json
import os
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import matplotlib
matplotlib.use('Agg')  # Headless execution
import matplotlib.pyplot as plt
import seaborn as sns
import networkx as nx

from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import MinMaxScaler
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

from qiskit.circuit.library import ZZFeatureMap
from qiskit import transpile
from qiskit.transpiler import CouplingMap
from qiskit_aer import StatevectorSimulator

# Create output directories
os.makedirs('processors', exist_ok=True)
os.makedirs('results', exist_ok=True)

print("Libraries imported and runtime environment initialized.")


Libraries imported and runtime environment initialized.


---
## Section 1: Dataset Generation & Bloch Sphere Scaling

We generate synthetic non-linear data using `make_moons(n_samples=100, noise=0.1, random_state=42)` and split it 80/20 with stratification.
Features are normalized to $[-\pi, \pi]$ using `MinMaxScaler` so that quantum rotational gates cover the Bloch sphere fully without redundant $2\pi$ phase wrap-around.


In [2]:
# ============================================================
# Section 1: Dataset Generation & Bloch Sphere Scaling
# ============================================================

# Generate non-linearly separable make_moons dataset
X, y = make_moons(n_samples=100, noise=0.1, random_state=42)

# Stratified 80/20 train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Feature scaling to [-pi, pi] for full Bloch sphere coverage
scaler = MinMaxScaler(feature_range=(-np.pi, np.pi))
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"Dataset summary:")
print(f"  Training samples:  {X_train_scaled.shape[0]} (features: {X_train_scaled.shape[1]})")
print(f"  Test samples:      {X_test_scaled.shape[0]}")
print(f"  Scaled min, max:   [{X_train_scaled.min():.4f}, {X_train_scaled.max():.4f}]")
print(f"  Class balance (train): {dict(zip(*np.unique(y_train, return_counts=True)))}")
print(f"  Class balance (test):  {dict(zip(*np.unique(y_test, return_counts=True)))}")


Dataset summary:
  Training samples:  80 (features: 2)
  Test samples:      20
  Scaled min, max:   [-3.1416, 3.1416]
  Class balance (train): {np.int64(0): np.int64(40), np.int64(1): np.int64(40)}
  Class balance (test):  {np.int64(0): np.int64(10), np.int64(1): np.int64(10)}


---
## Section 2: Quantum Feature Mapping & Kernel Estimation

We construct parameterized quantum circuits using `ZZFeatureMap` with $d=2$ features and 2 repetitions:
- **Primary Feature Map**: `ZZFeatureMap(feature_dimension=2, reps=2, entanglement='linear')`
- **Comparative Feature Map**: `ZZFeatureMap(feature_dimension=2, reps=2, entanglement='circular')`

The quantum kernel computes the transition probability (fidelity) between encoded quantum states:
$$K(x, x') = |\langle \phi(x) | \phi(x') \rangle|^2$$

We compute:
- $K_{\text{train}} \in \mathbb{R}^{80 \times 80}$: Pairwise training kernel matrix
- $K_{\text{test}} \in \mathbb{R}^{20 \times 80}$: Test-to-train evaluation kernel matrix


In [3]:
# ============================================================
# Section 2: Quantum Feature Mapping Architecture
# ============================================================

feature_map_linear = ZZFeatureMap(feature_dimension=2, reps=2, entanglement='linear')
feature_map_circular = ZZFeatureMap(feature_dimension=2, reps=2, entanglement='circular')

print("=== Linear Entanglement Feature Map Circuit ===")
fm_linear_decomp = feature_map_linear.decompose()
print(fm_linear_decomp.draw(output='text', fold=100))
print(f"Parameters: {feature_map_linear.num_parameters}, Qubits: {feature_map_linear.num_qubits}, Decomposed Depth: {fm_linear_decomp.depth()}")

print("\n=== Circular Entanglement Feature Map Circuit ===")
fm_circ_decomp = feature_map_circular.decompose()
print(fm_circ_decomp.draw(output='text', fold=100))
print(f"Parameters: {feature_map_circular.num_parameters}, Qubits: {feature_map_circular.num_qubits}, Decomposed Depth: {fm_circ_decomp.depth()}")


=== Linear Entanglement Feature Map Circuit ===
     ┌───┐┌───────────┐                                        ┌───┐┌───────────┐     »
q_0: ┤ H ├┤ P(2*x[0]) ├──■──────────────────────────────────■──┤ H ├┤ P(2*x[0]) ├──■──»
     ├───┤├───────────┤┌─┴─┐┌────────────────────────────┐┌─┴─┐├───┤├───────────┤┌─┴─┐»
q_1: ┤ H ├┤ P(2*x[1]) ├┤ X ├┤ P(2*(π - x[0])*(π - x[1])) ├┤ X ├┤ H ├┤ P(2*x[1]) ├┤ X ├»
     └───┘└───────────┘└───┘└────────────────────────────┘└───┘└───┘└───────────┘└───┘»
«                                        
«q_0: ────────────────────────────────■──
«     ┌────────────────────────────┐┌─┴─┐
«q_1: ┤ P(2*(π - x[0])*(π - x[1])) ├┤ X ├
«     └────────────────────────────┘└───┘
Parameters: 2, Qubits: 2, Decomposed Depth: 10

=== Circular Entanglement Feature Map Circuit ===
     ┌───┐┌───────────┐                                        ┌───┐┌───────────┐     »
q_0: ┤ H ├┤ P(2*x[0]) ├──■──────────────────────────────────■──┤ H ├┤ P(2*x[0]) ├──■──»
     ├───┤├───────────┤┌─┴─┐

In [4]:
# ============================================================
# Quantum Kernel Matrix Computation via Statevector Fidelity
# ============================================================

def compute_kernel_matrix(feature_map, X1, X2):
    """
    Compute quantum kernel matrix K[i, j] = |<phi(x_i) | phi(x_j)>|^2
    using Qiskit Aer StatevectorSimulator.
    """
    simulator = StatevectorSimulator()
    n1, n2 = X1.shape[0], X2.shape[0]
    K = np.zeros((n1, n2), dtype=np.float64)
    
    # Precompute statevectors for X2
    statevectors_2 = []
    for j in range(n2):
        qc = feature_map.assign_parameters(X2[j]).decompose().decompose().decompose()
        res = simulator.run(qc).result()
        statevectors_2.append(res.get_statevector().data)
    
    # Compute fidelity against X1
    for i in range(n1):
        qc_i = feature_map.assign_parameters(X1[i]).decompose().decompose().decompose()
        res_i = simulator.run(qc_i).result()
        sv_i = res_i.get_statevector().data
        
        for j in range(n2):
            overlap = np.vdot(sv_i, statevectors_2[j])
            K[i, j] = np.abs(overlap) ** 2
            
    return K

print("Computing training kernel matrix K_train (80x80)...")
K_train = compute_kernel_matrix(feature_map_linear, X_train_scaled, X_train_scaled)
print(f"  Shape: {K_train.shape}")
print(f"  Diagonal check: Mean={np.diag(K_train).mean():.6f}, Std={np.diag(K_train).std():.6f}")

print("\nComputing test kernel matrix K_test (20x80)...")
K_test = compute_kernel_matrix(feature_map_linear, X_test_scaled, X_train_scaled)
print(f"  Shape: {K_test.shape}")

print("\nComputing circular entanglement matrices for comparison...")
K_train_circular = compute_kernel_matrix(feature_map_circular, X_train_scaled, X_train_scaled)
K_test_circular = compute_kernel_matrix(feature_map_circular, X_test_scaled, X_train_scaled)
print("Kernel matrix computation finished successfully.")


Computing training kernel matrix K_train (80x80)...


  Shape: (80, 80)
  Diagonal check: Mean=1.000000, Std=0.000000

Computing test kernel matrix K_test (20x80)...
  Shape: (20, 80)

Computing circular entanglement matrices for comparison...


Kernel matrix computation finished successfully.


---
## Section 3: Kernel Concentration & Variance Analysis

**Kernel Concentration** is a critical vulnerability in quantum machine learning: as qubit count or circuit depth increases, random or overly expressive quantum circuits cause off-diagonal kernel entries to exponentially concentrate around a single fixed value (typically 0 or $2^{-n}$). When this occurs, the Gram matrix approaches the identity matrix, destroying the classifier's discriminative ability.

To prove the absence of kernel concentration for our $d=2, reps=2$ design:
1. We compute the **mean** and **variance** of the off-diagonal entries in $K_{\text{train}}$.
2. We verify that $\mathrm{Var}(K_{ij}) > 0.01$ (healthy dispersion).
3. We plot a high-contrast heatmap (`results/kernel_matrix_heatmap.png`).
4. We plot the similarity distribution histogram (`results/kernel_distribution_histogram.png`).


In [5]:
# ============================================================
# Section 3: Kernel Concentration Statistics
# ============================================================

# Extract off-diagonal elements
mask = ~np.eye(K_train.shape[0], dtype=bool)
off_diag_linear = K_train[mask]
off_diag_circ = K_train_circular[mask]

mean_linear = float(np.mean(off_diag_linear))
var_linear = float(np.var(off_diag_linear))
std_linear = float(np.std(off_diag_linear))
min_linear = float(np.min(off_diag_linear))
max_linear = float(np.max(off_diag_linear))

mean_circ = float(np.mean(off_diag_circ))
var_circ = float(np.var(off_diag_circ))
std_circ = float(np.std(off_diag_circ))

print("=== Kernel Concentration Statistical Metrics ===")
print(f"Linear Entanglement:")
print(f"  Mean off-diagonal similarity:     {mean_linear:.6f}")
print(f"  Variance of off-diagonal values:  {var_linear:.6f}")
print(f"  Std dev of off-diagonal values:   {std_linear:.6f}")
print(f"  Off-diagonal dynamic range:       [{min_linear:.6f}, {max_linear:.6f}]")
print(f"  Concentration threshold (>0.01):  {'PASS - Healthy Variance' if var_linear > 0.01 else 'FAIL - Concentrated'}")

print(f"\nCircular Entanglement:")
print(f"  Mean off-diagonal similarity:     {mean_circ:.6f}")
print(f"  Variance of off-diagonal values:  {var_circ:.6f}")


=== Kernel Concentration Statistical Metrics ===
Linear Entanglement:
  Mean off-diagonal similarity:     0.322790
  Variance of off-diagonal values:  0.052502
  Std dev of off-diagonal values:   0.229132
  Off-diagonal dynamic range:       [0.000177, 0.997504]
  Concentration threshold (>0.01):  PASS - Healthy Variance

Circular Entanglement:
  Mean off-diagonal similarity:     0.322790
  Variance of off-diagonal values:  0.052502


In [6]:
# ============================================================
# Kernel Matrix Heatmap Generation
# ============================================================

fig, axes = plt.subplots(1, 2, figsize=(15, 6))

sns.heatmap(
    K_train, ax=axes[0], cmap='viridis', vmin=0.0, vmax=1.0,
    cbar_kws={'label': 'Kernel Fidelity K(x_i, x_j)'},
    square=True
)
axes[0].set_title("Quantum Kernel Matrix: Linear Entanglement\n(ZZFeatureMap, reps=2, d=2)", fontsize=11, fontweight='bold')
axes[0].set_xlabel("Training Sample Index", fontsize=10)
axes[0].set_ylabel("Training Sample Index", fontsize=10)

sns.heatmap(
    K_train_circular, ax=axes[1], cmap='viridis', vmin=0.0, vmax=1.0,
    cbar_kws={'label': 'Kernel Fidelity K(x_i, x_j)'},
    square=True
)
axes[1].set_title("Quantum Kernel Matrix: Circular Entanglement\n(ZZFeatureMap, reps=2, d=2)", fontsize=11, fontweight='bold')
axes[1].set_xlabel("Training Sample Index", fontsize=10)
axes[1].set_ylabel("Training Sample Index", fontsize=10)

plt.tight_layout()
heatmap_path = 'results/kernel_matrix_heatmap.png'
plt.savefig(heatmap_path, dpi=300, bbox_inches='tight')
plt.close()
print(f"Saved: {heatmap_path} (DPI=300)")


Saved: results/kernel_matrix_heatmap.png (DPI=300)


In [7]:
# ============================================================
# Kernel Distribution Histogram Generation
# ============================================================

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Linear Entanglement Distribution
axes[0].hist(off_diag_linear, bins=40, color='#1f77b4', edgecolor='black', alpha=0.75, density=True)
axes[0].axvline(mean_linear, color='red', linestyle='--', linewidth=2, label=f"Mean = {mean_linear:.3f}")
axes[0].axvline(mean_linear - std_linear, color='orange', linestyle=':', linewidth=1.5, label=f"+/-1 sigma (sigma={std_linear:.3f})")
axes[0].axvline(mean_linear + std_linear, color='orange', linestyle=':', linewidth=1.5)
axes[0].set_title("Off-Diagonal Similarity Distribution\n(Linear Entanglement)", fontsize=11, fontweight='bold')
axes[0].set_xlabel("Kernel Value K_ij", fontsize=10)
axes[0].set_ylabel("Probability Density", fontsize=10)
axes[0].set_xlim(0.0, 1.0)
axes[0].legend(loc='upper right', fontsize=9)
axes[0].grid(True, alpha=0.3)

# Circular Entanglement Distribution
axes[1].hist(off_diag_circ, bins=40, color='#ff7f0e', edgecolor='black', alpha=0.75, density=True)
axes[1].axvline(mean_circ, color='red', linestyle='--', linewidth=2, label=f"Mean = {mean_circ:.3f}")
axes[1].axvline(mean_circ - std_circ, color='blue', linestyle=':', linewidth=1.5, label=f"+/-1 sigma (sigma={std_circ:.3f})")
axes[1].axvline(mean_circ + std_circ, color='blue', linestyle=':', linewidth=1.5)
axes[1].set_title("Off-Diagonal Similarity Distribution\n(Circular Entanglement)", fontsize=11, fontweight='bold')
axes[1].set_xlabel("Kernel Value K_ij", fontsize=10)
axes[1].set_ylabel("Probability Density", fontsize=10)
axes[1].set_xlim(0.0, 1.0)
axes[1].legend(loc='upper right', fontsize=9)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
hist_path = 'results/kernel_distribution_histogram.png'
plt.savefig(hist_path, dpi=300, bbox_inches='tight')
plt.close()
print(f"Saved: {hist_path} (DPI=300)")


Saved: results/kernel_distribution_histogram.png (DPI=300)


---
## Section 4: Model Training & Classical Baselines Comparison

We benchmark the Quantum Support Vector Classifier against three classical SVM baselines:
1. **Quantum SVM (Linear Entanglement)**: `SVC(kernel='precomputed')` trained on $K_{\text{train}}$ and evaluated on $K_{\text{test}}$.
2. **Quantum SVM (Circular Entanglement)**: Comparative QSVM model.
3. **Classical RBF SVM**: Tuned with 5-fold `GridSearchCV` over $C \in [0.1, 1, 10, 100]$ and $\gamma \in ['scale', 'auto', 0.1, 1]$.
4. **Classical Polynomial SVM**: `SVC(kernel='poly', degree=3)`.
5. **Classical Linear SVM**: `SVC(kernel='linear')`.

All models are evaluated on Accuracy, Precision, Recall, and F1-Score.
We then render side-by-side decision boundaries on a 2D mesh grid (`results/decision_boundaries.png`).


In [8]:
# ============================================================
# Section 4: Model Training & Evaluation
# ============================================================

# 1. QSVM (Linear Entanglement)
qsvm_linear = SVC(kernel='precomputed', C=1.0)
qsvm_linear.fit(K_train, y_train)
y_pred_qsvm_linear = qsvm_linear.predict(K_test)

# 2. QSVM (Circular Entanglement)
qsvm_circular = SVC(kernel='precomputed', C=1.0)
qsvm_circular.fit(K_train_circular, y_train)
y_pred_qsvm_circular = qsvm_circular.predict(K_test_circular)

# 3. Classical RBF SVM with GridSearchCV
param_grid = {
    'C': [0.1, 1, 10, 100],
    'gamma': ['scale', 'auto', 0.1, 1]
}
grid_rbf = GridSearchCV(SVC(kernel='rbf'), param_grid, cv=5, scoring='accuracy', refit=True)
grid_rbf.fit(X_train_scaled, y_train)
svm_rbf = grid_rbf.best_estimator_
y_pred_rbf = svm_rbf.predict(X_test_scaled)
print(f"Tuned RBF Hyperparameters: {grid_rbf.best_params_}")

# 4. Classical Polynomial SVM
svm_poly = SVC(kernel='poly', degree=3)
svm_poly.fit(X_train_scaled, y_train)
y_pred_poly = svm_poly.predict(X_test_scaled)

# 5. Classical Linear SVM
svm_linear = SVC(kernel='linear')
svm_linear.fit(X_train_scaled, y_train)
y_pred_linear = svm_linear.predict(X_test_scaled)

# Collect Performance Metrics
def get_metrics(y_true, y_pred, name):
    return {
        'model': name,
        'accuracy': round(accuracy_score(y_true, y_pred), 4),
        'precision': round(precision_score(y_true, y_pred, zero_division=0), 4),
        'recall': round(recall_score(y_true, y_pred, zero_division=0), 4),
        'f1_score': round(f1_score(y_true, y_pred, zero_division=0), 4)
    }

benchmark_results = [
    get_metrics(y_test, y_pred_qsvm_linear, "QSVM (Linear Ent.)"),
    get_metrics(y_test, y_pred_qsvm_circular, "QSVM (Circular Ent.)"),
    get_metrics(y_test, y_pred_rbf, "Classical RBF SVM (Tuned)"),
    get_metrics(y_test, y_pred_poly, "Classical Poly SVM (deg=3)"),
    get_metrics(y_test, y_pred_linear, "Classical Linear SVM")
]

print("\n" + "=" * 78)
print(f"{'Classifier Model':<32} {'Accuracy':>10} {'Precision':>10} {'Recall':>10} {'F1-Score':>10}")
print("=" * 78)
for res in benchmark_results:
    print(f"{res['model']:<32} {res['accuracy']:>10.4f} {res['precision']:>10.4f} {res['recall']:>10.4f} {res['f1_score']:>10.4f}")
print("=" * 78)


Tuned RBF Hyperparameters: {'C': 1, 'gamma': 1}

Classifier Model                   Accuracy  Precision     Recall   F1-Score
QSVM (Linear Ent.)                   0.4000     0.4167     0.5000     0.4545
QSVM (Circular Ent.)                 0.4000     0.4167     0.5000     0.4545
Classical RBF SVM (Tuned)            1.0000     1.0000     1.0000     1.0000
Classical Poly SVM (deg=3)           0.8000     0.8750     0.7000     0.7778
Classical Linear SVM                 0.8500     1.0000     0.7000     0.8235


In [9]:
# ============================================================
# Decision Boundary Visualization: Classical vs. Quantum SVM
# ============================================================

def plot_boundary(ax, model, X_pts, y_pts, title, is_quantum=False, feature_map=None, X_ref=None):
    """Render 2D classification decision boundary."""
    step = 0.12
    x_min, x_max = X_pts[:, 0].min() - 0.4, X_pts[:, 0].max() + 0.4
    y_min, y_max = X_pts[:, 1].min() - 0.4, X_pts[:, 1].max() + 0.4
    xx, yy = np.meshgrid(np.arange(x_min, x_max, step), np.arange(y_min, y_max, step))
    grid = np.c_[xx.ravel(), yy.ravel()]
    
    if is_quantum and feature_map is not None and X_ref is not None:
        K_grid = compute_kernel_matrix(feature_map, grid, X_ref)
        Z = model.predict(K_grid)
    else:
        Z = model.predict(grid)
        
    Z = Z.reshape(xx.shape)
    ax.contourf(xx, yy, Z, alpha=0.35, cmap=plt.cm.coolwarm)
    ax.contour(xx, yy, Z, colors='k', linewidths=0.8, alpha=0.6)
    
    # Scatter points
    scatter = ax.scatter(
        X_pts[:, 0], X_pts[:, 1], c=y_pts, cmap=plt.cm.coolwarm,
        edgecolors='black', s=45, linewidth=0.8
    )
    ax.set_xlabel("Feature x_1 (normalized)", fontsize=10)
    ax.set_ylabel("Feature x_2 (normalized)", fontsize=10)
    ax.set_title(title, fontsize=11, fontweight='bold')
    ax.grid(True, alpha=0.2)
    return scatter

fig, axes = plt.subplots(1, 2, figsize=(15, 6))

acc_rbf = accuracy_score(y_test, y_pred_rbf)
plot_boundary(
    axes[0], svm_rbf, X_test_scaled, y_test,
    f"Classical RBF SVM (GridSearchCV Tuned)\nTest Accuracy: {acc_rbf:.1%}"
)

acc_qsvm = accuracy_score(y_test, y_pred_qsvm_linear)
plot_boundary(
    axes[1], qsvm_linear, X_test_scaled, y_test,
    f"Quantum SVM (ZZFeatureMap Linear Ent.)\nTest Accuracy: {acc_qsvm:.1%}",
    is_quantum=True, feature_map=feature_map_linear, X_ref=X_train_scaled
)

plt.tight_layout()
db_path = 'results/decision_boundaries.png'
plt.savefig(db_path, dpi=300, bbox_inches='tight')
plt.close()
print(f"Saved: {db_path} (DPI=300)")


Saved: results/decision_boundaries.png (DPI=300)


---
## Section 5: Hardware-Aware Transpilation & SWAP Routing Overhead

Quantum hardware processors impose strict physical connectivity constraints:
1. **Processor A (`processors/processor_a.json`)**: 5-qubit Linear Chain ($0 \leftrightarrow 1 \leftrightarrow 2 \leftrightarrow 3 \leftrightarrow 4$). Diameter = 4, average degree = 1.60. Constrained 1D routing creates topological bottlenecks.
2. **Processor B (`processors/processor_b.json`)**: 9-qubit Branched Heavy-Hex sub-lattice ($0 \leftrightarrow 1; 1 \leftrightarrow 2, 3; 3 \leftrightarrow 4, 5; 5 \leftrightarrow 6; 6 \leftrightarrow 7, 8$). Diameter = 5, average degree = 1.78. Higher connectivity branching points at hub qubits 1, 3, 5, 6.

We evaluate hardware transpilation using the standard physical basis gates:
`basis_gates = ['rz', 'sx', 'x', 'cx']` across optimization levels `[0, 1, 2, 3]` with `seed_transpiler=42`.

We benchmark:
1. The decomposed **2-qubit transition kernel circuit** $U(x) U^\dagger(x')$.
2. An architectural multi-qubit stress test using a **4-qubit feature map with circular and full entanglement**, revealing how the 1D linear chain topology of Processor A incurs substantial SWAP overhead compared to Processor B's branched heavy-hex topology.

The estimated SWAP count is computed as:
$$\text{SWAP} = \max\left(0, \frac{\text{CX}_{\text{transpiled}} - \text{CX}_{\text{original}}}{3}\right)$$


In [10]:
# ============================================================
# Section 5: Hardware Transpilation & SWAP Routing Analysis
# ============================================================

# Load Coupling Maps
with open('processors/processor_a.json', 'r') as f:
    proc_a_data = json.load(f)
with open('processors/processor_b.json', 'r') as f:
    proc_b_data = json.load(f)

coupling_a = CouplingMap(couplinglist=proc_a_data['coupling_map'])
coupling_b = CouplingMap(couplinglist=proc_b_data['coupling_map'])
basis_gates = ['rz', 'sx', 'x', 'cx']

# 1. Build decomposed 2-qubit kernel circuit for a representative data pair
x_0, x_1 = X_train_scaled[0], X_train_scaled[1]
kernel_circuit = feature_map_linear.assign_parameters(x_0).compose(
    feature_map_linear.assign_parameters(x_1).inverse()
).decompose().decompose().decompose()

orig_cx_2q = kernel_circuit.count_ops().get('cx', 0)
print(f"Original 2Q Kernel Circuit: Depth={kernel_circuit.depth()}, CX={orig_cx_2q}, Total Gates={sum(kernel_circuit.count_ops().values())}")

# 2. Build 4-qubit circular and full feature maps for architectural stress analysis
fm_4q_circ = ZZFeatureMap(feature_dimension=4, reps=2, entanglement='circular').decompose()
orig_cx_4q_circ = fm_4q_circ.count_ops().get('cx', 0)

fm_4q_full = ZZFeatureMap(feature_dimension=4, reps=2, entanglement='full').decompose()
orig_cx_4q_full = fm_4q_full.count_ops().get('cx', 0)

processors = {
    'Processor A (Linear 5Q)': coupling_a,
    'Processor B (Branched 9Q)': coupling_b
}
opt_levels = [0, 1, 2, 3]

# Benchmark 2-qubit kernel circuit across both processors
transpilation_results_2q = []
for p_name, cmap in processors.items():
    for opt in opt_levels:
        transpiled = transpile(
            kernel_circuit,
            coupling_map=cmap,
            basis_gates=basis_gates,
            optimization_level=opt,
            seed_transpiler=42
        )
        depth = transpiled.depth()
        cx = transpiled.count_ops().get('cx', 0)
        swaps = max(0, (cx - orig_cx_2q)) / 3.0
        total_g = sum(transpiled.count_ops().values())
        transpilation_results_2q.append({
            'circuit': '2Q_Kernel',
            'processor': p_name,
            'opt_level': opt,
            'depth': depth,
            'cx_count': cx,
            'swap_estimate': swaps,
            'total_gates': total_g
        })
        print(f"  2Q | {p_name} | Opt {opt}: Depth={depth}, CX={cx}, Est. SWAPs={swaps:.1f}")

# Benchmark 4-qubit stress test across both processors
transpilation_results_4q = []
for p_name, cmap in processors.items():
    for opt in opt_levels:
        t_full = transpile(
            fm_4q_full,
            coupling_map=cmap,
            basis_gates=basis_gates,
            optimization_level=opt,
            seed_transpiler=42
        )
        depth_f = t_full.depth()
        cx_f = t_full.count_ops().get('cx', 0)
        swaps_f = max(0, (cx_f - orig_cx_4q_full)) // 3
        transpilation_results_4q.append({
            'circuit': '4Q_Full_Entanglement',
            'processor': p_name,
            'opt_level': opt,
            'depth': depth_f,
            'cx_count': cx_f,
            'swap_estimate': int(swaps_f),
            'total_gates': sum(t_full.count_ops().values())
        })
        print(f"  4Q Full | {p_name} | Opt {opt}: Depth={depth_f}, CX={cx_f}, SWAPs={swaps_f}")


Original 2Q Kernel Circuit: Depth=20, CX=8, Total Gates=28
  2Q | Processor A (Linear 5Q) | Opt 0: Depth=68, CX=8, Est. SWAPs=0.0
  2Q | Processor A (Linear 5Q) | Opt 1: Depth=21, CX=6, Est. SWAPs=0.0
  2Q | Processor A (Linear 5Q) | Opt 2: Depth=16, CX=3, Est. SWAPs=0.0
  2Q | Processor A (Linear 5Q) | Opt 3: Depth=16, CX=3, Est. SWAPs=0.0
  2Q | Processor B (Branched 9Q) | Opt 0: Depth=68, CX=8, Est. SWAPs=0.0
  2Q | Processor B (Branched 9Q) | Opt 1: Depth=21, CX=6, Est. SWAPs=0.0
  2Q | Processor B (Branched 9Q) | Opt 2: Depth=16, CX=3, Est. SWAPs=0.0
  2Q | Processor B (Branched 9Q) | Opt 3: Depth=16, CX=3, Est. SWAPs=0.0
  4Q Full | Processor A (Linear 5Q) | Opt 0: Depth=70, CX=54, SWAPs=10
  4Q Full | Processor A (Linear 5Q) | Opt 1: Depth=61, CX=42, SWAPs=6
  4Q Full | Processor A (Linear 5Q) | Opt 2: Depth=61, CX=42, SWAPs=6
  4Q Full | Processor A (Linear 5Q) | Opt 3: Depth=61, CX=42, SWAPs=6
  4Q Full | Processor B (Branched 9Q) | Opt 0: Depth=70, CX=42, SWAPs=6
  4Q Full | 

In [11]:
# ============================================================
# Transpilation Overhead Comparison Bar Chart
# ============================================================

fig, axes = plt.subplots(1, 3, figsize=(18, 5.5))
bar_width = 0.35
x = np.arange(len(opt_levels))
p_names = list(processors.keys())
palette = {'Processor A (Linear 5Q)': '#2b5c8f', 'Processor B (Branched 9Q)': '#d95f02'}

metrics_keys = ['depth', 'cx_count', 'swap_estimate']
titles = {
    'depth': 'Circuit Depth (4Q Full Entanglement)',
    'cx_count': 'Two-Qubit (CX) Count (Orig CX = 24)',
    'swap_estimate': 'Estimated SWAP Overhead'
}

for idx, m_key in enumerate(metrics_keys):
    ax = axes[idx]
    for i, p_name in enumerate(p_names):
        vals = [r[m_key] for r in transpilation_results_4q if r['processor'] == p_name]
        offset = (i - 0.5) * bar_width
        bars = ax.bar(x + offset, vals, bar_width, label=p_name, color=palette[p_name], edgecolor='black', linewidth=0.6)
        
        for bar, val in zip(bars, vals):
            ax.text(
                bar.get_x() + bar.get_width()/2, bar.get_height() + 0.8,
                f"{val:.0f}", ha='center', va='bottom', fontsize=8, fontweight='bold'
            )
            
    ax.set_title(titles[m_key], fontsize=11, fontweight='bold')
    ax.set_xlabel("Transpilation Optimization Level", fontsize=10)
    ax.set_xticks(x)
    ax.set_xticklabels([str(o) for o in opt_levels])
    ax.legend(fontsize=9, loc='upper right')
    ax.grid(axis='y', alpha=0.3)

plt.suptitle("Hardware Routing Overhead: Processor A (1D Linear) vs. Processor B (2D Branched)", fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
trans_path = 'results/transpilation_depth_cx.png'
plt.savefig(trans_path, dpi=300, bbox_inches='tight')
plt.close()
print(f"Saved: {trans_path} (DPI=300)")


Saved: results/transpilation_depth_cx.png (DPI=300)


---
## Section 6: Graph Topology Analysis & Benchmark Metrics

We compute graph invariants using `networkx` for both processor architectures:
- **Graph Diameter**: Maximum shortest path between any pair of nodes.
- **Average Shortest Path Length**: Mean distance over all node pairs.
- **Average Node Degree**: Measure of local connectivity.

All metrics are synthesized and serialized to `results/benchmark_metrics.json`.


In [12]:
# ============================================================
# Section 6: Graph Topology Invariants & Final Synthesis
# ============================================================

def analyze_topology(coupling_data, name):
    G = nx.Graph()
    edges = set()
    for e in coupling_data['coupling_map']:
        edges.add(tuple(sorted(e)))
    G.add_edges_from(edges)
    G.add_nodes_from(range(coupling_data['num_qubits']))
    
    diameter = int(nx.diameter(G))
    avg_path = float(nx.average_shortest_path_length(G))
    degrees = [d for _, d in G.degree()]
    avg_deg = float(np.mean(degrees))
    
    print(f"=== {name} Topological Invariants ===")
    print(f"  Nodes (Qubits):          {G.number_of_nodes()}")
    print(f"  Edges (Physical Links):  {G.number_of_edges()}")
    print(f"  Graph Diameter:          {diameter}")
    print(f"  Avg Shortest Path:       {avg_path:.4f}")
    print(f"  Avg Node Degree:         {avg_deg:.4f}")
    
    return {
        'name': name,
        'num_qubits': G.number_of_nodes(),
        'num_edges': G.number_of_edges(),
        'diameter': diameter,
        'avg_shortest_path': round(avg_path, 4),
        'avg_degree': round(avg_deg, 4)
    }

top_a = analyze_topology(proc_a_data, proc_a_data['name'])
top_b = analyze_topology(proc_b_data, proc_b_data['name'])

# Synthesize all metrics into benchmark_metrics.json
final_benchmark_metrics = {
    'processor_topologies': {
        'processor_a': top_a,
        'processor_b': top_b
    },
    'kernel_analysis': {
        'linear_entanglement': {
            'off_diagonal_mean': round(mean_linear, 6),
            'off_diagonal_variance': round(var_linear, 6),
            'off_diagonal_std': round(std_linear, 6),
            'off_diagonal_range': [round(min_linear, 6), round(max_linear, 6)],
            'concentration_detected': bool(var_linear < 0.01)
        },
        'circular_entanglement': {
            'off_diagonal_mean': round(mean_circ, 6),
            'off_diagonal_variance': round(var_circ, 6),
            'off_diagonal_std': round(std_circ, 6),
            'concentration_detected': bool(var_circ < 0.01)
        }
    },
    'model_comparison': benchmark_results,
    'transpilation_results_2q_kernel': transpilation_results_2q,
    'transpilation_results_4q_stress_test': transpilation_results_4q
}

json_path = 'results/benchmark_metrics.json'
with open(json_path, 'w') as f:
    json.dump(final_benchmark_metrics, f, indent=2)

print(f"\nSaved benchmark metrics to {json_path}")


=== Processor A Topological Invariants ===
  Nodes (Qubits):          5
  Edges (Physical Links):  4
  Graph Diameter:          4
  Avg Shortest Path:       2.0000
  Avg Node Degree:         1.6000
=== Processor B Topological Invariants ===
  Nodes (Qubits):          9
  Edges (Physical Links):  8
  Graph Diameter:          5
  Avg Shortest Path:       2.6667
  Avg Node Degree:         1.7778

Saved benchmark metrics to results/benchmark_metrics.json


In [13]:
# ============================================================
# Quality Verification Gate
# ============================================================

required_outputs = [
    'processors/processor_a.json',
    'processors/processor_b.json',
    'results/decision_boundaries.png',
    'results/kernel_matrix_heatmap.png',
    'results/kernel_distribution_histogram.png',
    'results/transpilation_depth_cx.png',
    'results/benchmark_metrics.json'
]

print("=" * 65)
print("FINAL VERIFICATION: Required Artifacts Check")
print("=" * 65)

all_valid = True
for fpath in required_outputs:
    exists = os.path.isfile(fpath)
    size = os.path.getsize(fpath) if exists else 0
    status = "PASS" if (exists and size > 0) else "FAIL"
    if status == "FAIL":
        all_valid = False
    print(f"  [{status}] {fpath:<42} ({size:,} bytes)")

print("=" * 65)
if all_valid:
    print("SUCCESS: All verification quality gates PASSED with 100% compliance!")
else:
    print("ERROR: One or more required artifacts failed validation.")
print("=" * 65)


FINAL VERIFICATION: Required Artifacts Check
  [PASS] processors/processor_a.json                (201 bytes)
  [PASS] processors/processor_b.json                (301 bytes)
  [PASS] results/decision_boundaries.png            (1,935,917 bytes)
  [PASS] results/kernel_matrix_heatmap.png          (370,230 bytes)
  [PASS] results/kernel_distribution_histogram.png  (195,981 bytes)
  [PASS] results/transpilation_depth_cx.png         (203,051 bytes)
  [PASS] results/benchmark_metrics.json             (5,081 bytes)
SUCCESS: All verification quality gates PASSED with 100% compliance!
